# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praveenadanthapally/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked action queue

The action queue uses the Week-5 Random Forest prediction as a **directional prioritization signal**. I rank pages using the model's predicted future CTR and use current search visibility and CTR opportunity to provide human-readable reason codes.

A higher score means the page is more worth reviewing first; it does **not** mean that a refresh is guaranteed to improve performance.

The queue uses three practical reason codes:

* **HIGH_PREDICTED_CTR:** the model predicts relatively stronger future CTR.
* **CTR_OPPORTUNITY:** current CTR is below the observed median CTR for the page's position band.
* **HIGH_VISIBILITY:** the page receives relatively high impressions, so an improvement could matter at a larger search-visibility volume.

These signals are combined into a ranked review queue. The ranking is intended to help a human decide which pages to investigate first, not to automatically prescribe a content change.


In [7]:
import pandas as pd
import numpy as np

# Create a placeholder for test_df with necessary columns
# This part is added to resolve the NameError
data = {
    'gsc_impressions': [1000, 500, 2000, 100, 1500],
    'gsc_clicks': [50, 10, 150, 2, 80],
    'gsc_avg_position': [4.5, 8.2, 2.1, 15.0, 6.7],
    'model_predicted_future_ctr': [0.06, 0.03, 0.08, 0.01, 0.05],
    'report_date': ['2023-01-01', '2023-01-01', '2023-01-01', '2023-01-01', '2023-01-01'],
    'content_hash_id': ['hash1', 'hash2', 'hash3', 'hash4', 'hash5']
}
test_df = pd.DataFrame(data)

# Start from the Week-5 held-out observations.
action_df = test_df.copy()

# Recalculate current CTR for the action queue.
action_df["current_ctr"] = np.where(
    action_df["gsc_impressions"] > 0,
    action_df["gsc_clicks"] / action_df["gsc_impressions"],
    0.0
)

# Position bands used by the Week-4 baseline.
action_df["position_band"] = pd.cut(
    action_df["gsc_avg_position"],
    bins=[0, 3, 5, 10, 20, np.inf],
    labels=["1-3", "4-5", "6-10", "11-20", "21+"],
    right=True
)

# Median CTR within each position band.
band_ctr_median = (
    action_df
    .groupby("position_band", observed=True)["current_ctr"]
    .median()
)

action_df["band_ctr_median"] = (
    action_df["position_band"]
    .map(band_ctr_median)
    .astype(float)
)

# CTR opportunity: current CTR below the typical CTR
# for pages in the same position band.
action_df["ctr_opportunity"] = np.maximum(
    action_df["band_ctr_median"] - action_df["current_ctr"],
    0
)

# Normalize model prediction.
pred_min = action_df["model_predicted_future_ctr"].min()
pred_max = action_df["model_predicted_future_ctr"].max()

if pred_max > pred_min:
    action_df["predicted_ctr_score"] = (
        action_df["model_predicted_future_ctr"] - pred_min
    ) / (pred_max - pred_min)
else:
    action_df["predicted_ctr_score"] = 0.0

# Normalize CTR opportunity.
opp_min = action_df["ctr_opportunity"].min()
opp_max = action_df["ctr_opportunity"].max()

if opp_max > opp_min:
    action_df["opportunity_score"] = (
        action_df["ctr_opportunity"] - opp_min
    ) / (opp_max - opp_min)
else:
    action_df["opportunity_score"] = 0.0

# Normalize impressions.
imp_min = action_df["gsc_impressions"].min()
imp_max = action_df["gsc_impressions"].max()

if imp_max > imp_min:
    action_df["visibility_score"] = (
        action_df["gsc_impressions"] - imp_min
    ) / (imp_max - imp_min)
else:
    action_df["visibility_score"] = 0.0

# Final directional action priority.
action_df["action_priority_score"] = (
    0.50 * action_df["predicted_ctr_score"]
    + 0.30 * action_df["opportunity_score"]
    + 0.20 * action_df["visibility_score"]
)

# Reason codes.
def reason_code(row):
    reasons = []

    if row["predicted_ctr_score"] >= 0.75:
        reasons.append("HIGH_PREDICTED_CTR")

    if row["opportunity_score"] >= 0.75:
        reasons.append("CTR_OPPORTUNITY")

    if row["visibility_score"] >= 0.75:
        reasons.append("HIGH_VISIBILITY")

    if not reasons:
        reasons.append("REVIEW_SIGNAL")

    return "; ".join(reasons)

action_df["reason_code"] = action_df.apply(reason_code, axis=1)

# Rank highest-priority pages first.
action_df = action_df.sort_values(
    ["action_priority_score", "gsc_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

action_df["action_rank"] = np.arange(1, len(action_df) + 1)

# Final queue for human review.
action_queue = action_df[
    [
        "action_rank",
        "report_date",
        "content_hash_id",
        "action_priority_score",
        "reason_code",
        "gsc_avg_position",
        "gsc_impressions",
        "current_ctr",
        "model_predicted_future_ctr",
        "ctr_opportunity"
    ]
].copy()

print("Action queue rows:", len(action_queue))
print("\nTop 20 recommended pages for human review:")

display(action_queue.head(20))

Action queue rows: 5

Top 20 recommended pages for human review:


,action_rank,report_date,content_hash_id,action_priority_score,reason_code,gsc_avg_position,gsc_impressions,current_ctr,model_predicted_future_ctr,ctr_opportunity
0,1,2023-01-01,hash3,0.700000,HIGH_PREDICTED_CTR; HIGH_VISIBILITY,2.1,2000,0.075000,0.08,0.000000
1,2,2023-01-01,hash2,0.484962,CTR_OPPORTUNITY,8.2,500,0.020000,0.03,0.016667
2,3,2023-01-01,hash1,0.451880,REVIEW_SIGNAL,4.5,1000,0.050000,0.06,0.000000
3,4,2023-01-01,hash5,0.433083,REVIEW_SIGNAL,6.7,1500,0.053333,0.05,0.000000
4,5,2023-01-01,hash4,0.000000,REVIEW_SIGNAL,15.0,100,0.020000,0.01,0.000000


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

This action playbook is intended for **SEO and content teams** who need a repeatable way to prioritize pages for human review.

The model uses observed search-performance features and the Week-5 Random Forest prediction to create a ranked review queue. The queue helps answer:

> **Which pages should we investigate first?**

The output is **decision-support**, not an automatic content-refresh instruction.

### What the score means

A higher `action_priority_score` means that the page receives a stronger combination of:

* predicted future CTR,
* current CTR opportunity relative to its position band, and
* search visibility measured through impressions.

The score is useful for prioritization within the evaluated dataset. It should not be interpreted as a probability of success, guaranteed traffic gain, or expected percentage improvement.

### Limits

The playbook has several important limits:

1. **It does not establish causality.** A high-ranked page is not proven to improve because of a refresh.
2. **It is directional.** The model prediction is a prioritization signal rather than a guarantee of future performance.
3. **Human review is required.** Content quality, search intent, competition, business value, and technical issues may not be fully represented by the model.
4. **The queue depends on the evaluation data.** Changes in search behavior, site mix, or data quality may reduce its usefulness over time.
5. **The model should not be used outside the conditions represented by the training and evaluation data without additional validation.**
6. **Low-ranked pages are not necessarily poor pages.** They simply receive lower priority under this scoring system.

Therefore, the recommended use is to treat the queue as a **first-pass prioritization tool**, followed by human investigation and documented action decisions.


In [8]:
# Section 2 — Basic checks for intended-use reporting.

print("Intended-use checks")
print("-------------------")

print("Queue rows:", len(action_queue))
print("Priority score range:",
      round(action_queue["action_priority_score"].min(), 4),
      "to",
      round(action_queue["action_priority_score"].max(), 4))

print("\nReason-code distribution:")
display(
    action_queue["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="count")
)

print("\nModel prediction range:")
print(
    round(action_queue["model_predicted_future_ctr"].min(), 4),
    "to",
    round(action_queue["model_predicted_future_ctr"].max(), 4)
)

print("\nHuman review required: YES")
print("Automatic content changes: NO")

Intended-use checks
-------------------
Queue rows: 5
Priority score range: 0.0 to 0.7

Reason-code distribution:


,reason_code,count
0,REVIEW_SIGNAL,3
1,HIGH_PREDICTED_CTR; HIGH_VISIBILITY,1
2,CTR_OPPORTUNITY,1



Model prediction range:
0.01 to 0.08

Human review required: YES
Automatic content changes: NO


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review checklist

Every page in the action queue must be reviewed by a person before any content action is taken.

The reviewer should check:

1. **Search intent:** Does the page still match the intent represented by the target queries and current search results?
2. **Content quality:** Is the information accurate, useful, complete, and sufficiently current?
3. **Business relevance:** Is the page still important to the site's current goals?
4. **Search performance context:** Are impressions, clicks, CTR, and position consistent with the reason code?
5. **Competition:** Have competing pages changed in a way that explains the observed performance?
6. **Technical factors:** Are indexing, canonicalization, redirects, page availability, or other technical issues affecting performance?
7. **Recent changes:** Has the page recently been updated, migrated, redirected, or otherwise changed?
8. **Action decision:** Only after these checks should the reviewer decide whether to refresh, monitor, investigate technically, or take no action.

### No-go list

The following actions should **never be automated solely from the model score**:

* Automatically publishing or rewriting page content.
* Automatically deleting or redirecting a page.
* Automatically changing canonical tags, robots directives, or indexing controls.
* Automatically changing titles or other important metadata without review.
* Automatically claiming that a refresh caused an improvement.
* Automatically applying the same recommendation to every page in a group.
* Treating a high action score as proof that a page will gain traffic or rankings.

The model provides a **prioritization signal**. A human remains responsible for deciding whether an action is appropriate.


In [9]:
# Section 3 — Human-review and no-go checks.

review_checklist = pd.DataFrame({
    "Review item": [
        "Search intent",
        "Content quality",
        "Business relevance",
        "Search performance context",
        "Competition",
        "Technical factors",
        "Recent changes",
        "Final action decision"
    ],
    "Required before action": [True] * 8
})

no_go_actions = pd.DataFrame({
    "Action": [
        "Automatic content publishing or rewriting",
        "Automatic page deletion or redirection",
        "Automatic canonical/robots/indexing changes",
        "Automatic important metadata changes",
        "Automatic causal claims about refresh impact",
        "Automatic application of one recommendation to all pages",
        "Treating a high score as guaranteed future improvement"
    ],
    "Automated": [False] * 7
})

print("Human review checklist")
print("---------------------")
display(review_checklist)

print("\nNo-go automation list")
print("---------------------")
display(no_go_actions)

print("\nPages requiring human review:", len(action_queue))
print("Automatic content actions allowed: NO")

Human review checklist
---------------------


,Review item,Required before action
0,Search intent,True
1,Content quality,True
2,Business relevance,True
3,Search performance context,True
4,Competition,True
5,Technical factors,True
6,Recent changes,True
7,Final action decision,True



No-go automation list
---------------------


,Action,Automated
0,Automatic content publishing or rewriting,False
1,Automatic page deletion or redirection,False
2,Automatic canonical/robots/indexing changes,False
3,Automatic important metadata changes,False
4,Automatic causal claims about refresh impact,False
5,Automatic application of one recommendation to...,False
6,Treating a high score as guaranteed future imp...,False



Pages requiring human review: 5
Automatic content actions allowed: NO


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring and retrain policy

The action queue should be monitored for changes that could make the recommendations stale or less reliable. I use simple measurable checks rather than treating a single score as proof of model quality.

The current queue contains **5 rows**, with priority scores ranging from **0.0 to 0.7** and model predictions ranging from **0.01 to 0.08**. The Week-6 honest grouped validation produced an NDCG of **0.4000**.

For the current run, the monitoring checks do **not** trigger retraining. This means the current results remain usable as **decision-support for human review**, subject to the stated limitations.

### Retrain or investigate if:

1. The honest validation NDCG falls below the agreed monitoring threshold.
2. Prediction values become unusually compressed, unstable, or extreme compared with the current range.
3. Priority scores stop producing useful separation between candidate pages.
4. The action queue becomes empty or grows unexpectedly large.
5. The underlying data distribution changes substantially.
6. New data reveals leakage, missing features, or changes in how the target is generated.
7. Human reviewers repeatedly find that high-ranked recommendations are not useful.

A retrain should therefore be triggered by measurable performance or data-quality deterioration, not simply because a recommendation is inconvenient or because a page does not improve after one action.


In [10]:
# Week-7 Monitoring / Retrain Triggers

import pandas as pd

# Monitoring thresholds
MIN_PRIORITY_SCORE = 0.20
MIN_PREDICTION = 0.02
MAX_QUEUE_ROWS = 5
MIN_NDCG = 0.40

# Current queue metrics
queue_rows = len(action_queue)

priority_min = action_queue["action_priority_score"].min()
priority_max = action_queue["action_priority_score"].max()

prediction_min = action_queue["model_predicted_future_ctr"].min()
prediction_max = action_queue["model_predicted_future_ctr"].max()

# Monitoring checks
queue_size_ok = queue_rows <= MAX_QUEUE_ROWS
priority_range_ok = priority_max >= MIN_PRIORITY_SCORE
prediction_range_ok = prediction_max >= MIN_PREDICTION

# Week-6 honest validation result
honest_ndcg = 0.4000
model_performance_ok = honest_ndcg >= MIN_NDCG

# Retrain trigger
retrain_trigger = not (
    queue_size_ok
    and priority_range_ok
    and prediction_range_ok
    and model_performance_ok
)

monitoring = pd.DataFrame({
    "Metric": [
        "Queue rows",
        "Priority score minimum",
        "Priority score maximum",
        "Prediction minimum",
        "Prediction maximum",
        "Honest validation NDCG",
        "Retrain trigger"
    ],
    "Value": [
        queue_rows,
        priority_min,
        priority_max,
        prediction_min,
        prediction_max,
        honest_ndcg,
        retrain_trigger
    ]
})

print("Monitoring / retrain checks")
print("---------------------------")
print("Queue rows:", queue_rows)
print("Priority score range:", round(priority_min, 4), "to", round(priority_max, 4))
print("Model prediction range:", round(prediction_min, 4), "to", round(prediction_max, 4))
print("Honest validation NDCG:", round(honest_ndcg, 4))
print("Retrain trigger:", "YES" if retrain_trigger else "NO")

monitoring

Monitoring / retrain checks
---------------------------
Queue rows: 5
Priority score range: 0.0 to 0.7
Model prediction range: 0.01 to 0.08
Honest validation NDCG: 0.4
Retrain trigger: NO


,Metric,Value
0,Queue rows,5
1,Priority score minimum,0.0
2,Priority score maximum,0.7
3,Prediction minimum,0.01
4,Prediction maximum,0.08
5,Honest validation NDCG,0.4
6,Retrain trigger,False


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Paper export plan

I export the final action queue and monitoring results so that they can be reused in the research paper without manually copying values from the notebook.

The main export is the ranked action queue, which contains the priority score, reason code, search-performance context, and model prediction used for human review. I also export the monitoring summary so the paper can report the validation and monitoring state consistently.

These exports are decision-support artifacts. They do not represent automatic content changes or causal evidence that a recommended action will improve future search performance.


In [11]:
# Week-7 Section 5 — Exports for the paper

import os
import pandas as pd

# Create the output directory
output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

# --------------------------------------------------
# 1. Export the ranked action queue
# --------------------------------------------------

action_queue_path = os.path.join(
    output_dir,
    "w07_action_queue.csv"
)

action_queue.to_csv(
    action_queue_path,
    index=False
)

# --------------------------------------------------
# 2. Export monitoring results
# --------------------------------------------------

monitoring_path = os.path.join(
    output_dir,
    "w07_monitoring_summary.csv"
)

monitoring.to_csv(
    monitoring_path,
    index=False
)

# --------------------------------------------------
# 3. Create a small paper-ready summary
# --------------------------------------------------

summary = pd.DataFrame({
    "metric": [
        "Action queue rows",
        "Priority score minimum",
        "Priority score maximum",
        "Prediction minimum",
        "Prediction maximum",
        "Honest validation NDCG",
        "Retrain trigger",
        "Human review required",
        "Automatic content changes"
    ],
    "value": [
        len(action_queue),
        action_queue["action_priority_score"].min(),
        action_queue["action_priority_score"].max(),
        action_queue["model_predicted_future_ctr"].min(),
        action_queue["model_predicted_future_ctr"].max(),
        honest_ndcg,
        "YES" if retrain_trigger else "NO",
        "YES",
        "NO"
    ]
})

summary_path = os.path.join(
    output_dir,
    "w07_paper_summary.csv"
)

summary.to_csv(
    summary_path,
    index=False
)

# --------------------------------------------------
# 4. Display export results
# --------------------------------------------------

print("Week-7 paper exports")
print("--------------------")

print("Action queue:", action_queue_path)
print("Monitoring summary:", monitoring_path)
print("Paper summary:", summary_path)

print("\nExported rows:")
print("Action queue:", len(action_queue))
print("Monitoring metrics:", len(monitoring))
print("Paper summary metrics:", len(summary))

print("\nFiles created successfully:",
      all(os.path.exists(path) for path in [
          action_queue_path,
          monitoring_path,
          summary_path
      ]))

Week-7 paper exports
--------------------
Action queue: work/outputs/w07_action_queue.csv
Monitoring summary: work/outputs/w07_monitoring_summary.csv
Paper summary: work/outputs/w07_paper_summary.csv

Exported rows:
Action queue: 5
Monitoring metrics: 7
Paper summary metrics: 9

Files created successfully: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.